# Gelişmiş NLP Görevleri

Table of Contents:



## 1. Soru Cevaplama (Question Answering):

### 1.a. Bert Modeli:

In [3]:
from transformers import BertTokenizer, BertForQuestionAnswering
import torch

import warnings
warnings.filterwarnings("ignore")

model_name = "bert-large-uncased-whole-word-masking-finetuned-squad"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForQuestionAnswering.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking-finetuned-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def predict_answer_bert(question, context):
    encoding = tokenizer(
        question,
        context,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=False
        )

    start_scores, end_scores = outputs

    start_index = torch.argmax(start_scores, dim=1).item()
    end_index = torch.argmax(end_scores, dim=1).item()

    answer_tokens = tokenizer.convert_ids_to_tokens(
        input_ids[0][start_index:end_index + 1]
    )

    return tokenizer.convert_tokens_to_string(answer_tokens)

In [ ]:
question = "Who signed the Declaration of Independence?"
context = "The Declaration of Independence was signed on July 4, 1776, by representatives of the thirteen American colonies. It marked the colonies' assertion of independence from British rule and laid the foundation for the United States of America."

predict_answer_bert(question, context)

'representatives of the thirteen american colonies'

### 1.b. GPT Modeli:

In [9]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [10]:
def predict_answer_gpt2(context, question):
    input_text = f"Context: {context}\nQuestion: {question}\nAnswer:"
    input_ids = tokenizer.encode(input_text, return_tensors="pt")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_length=100,
            num_return_sequences=1,
            no_repeat_ngram_size=2,
            early_stopping=True
        )

    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return answer.split("Answer:")[-1].strip()

In [14]:
question = "When was the declaration of Independence signed?"
context = "The Declaration of Independence was signed on July 4, 1776, by representatives of the thirteen American colonies. It marked the colonies' assertion of independence from British rule and laid the foundation for the United States of America."

predict_answer_gpt2(context, question)

'On July 3, 1801, the Declaration was ratified by the states. The United Kingdom of Great Britain and Ireland signed the treaty of peace on June 30, 1830. On June 29, 1840, President'